# Prompt Engineering Mastery for Customer Scenarios Interview

**Goal**: Build the prompt engineering muscle memory you need for the 50-minute Customer Scenarios interview. By the end, you'll be able to:
- Quickly craft effective prompts under time pressure
- Apply Anthropic's recommended prompt engineering techniques
- Iteratively improve a prompt using a systematic toolkit
- Structure prompts using XML tags (Claude's preferred format)
- Chain prompts for complex multi-step tasks

**Interview context**: You'll be in Google Colab with the Anthropic SDK pre-installed. You'll get a customer scenario and need to build a working solution. Prompt quality is often the difference between a solution that works and one that doesn't.

---

## Setup

In [ ]:
!pip install anthropic -q

In [ ]:
import anthropic
import json

# Option 1: Google Colab pattern (what the interview will likely use)
# from google.colab import userdata
# client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

# Option 2: Environment variable fallback (local dev)
# export ANTHROPIC_API_KEY=sk-ant-...
client = anthropic.Anthropic()

MODEL = "claude-haiku-4-5-20251001"

def ask(prompt, system=None, max_tokens=1024):
    """Quick helper to send a prompt and get text back."""
    kwargs = {"model": MODEL, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system
    response = client.messages.create(**kwargs)
    return response.content[0].text

def compare(label_a, prompt_a, label_b, prompt_b, system=None, max_tokens=1024):
    """Run two prompts side-by-side and print results."""
    print(f"={'=' * 60}")
    print(f"  {label_a}")
    print(f"={'=' * 60}")
    print(ask(prompt_a, system=system, max_tokens=max_tokens))
    print(f"\n{'=' * 60}")
    print(f"  {label_b}")
    print(f"={'=' * 60}")
    print(ask(prompt_b, system=system, max_tokens=max_tokens))

print("Client ready. Model:", MODEL)

---

## 1.1 Clear and Direct Instructions

The single most impactful technique. Most prompts fail because they are **vague**, not because they are wrong.

### The principle
Treat Claude like a brilliant new hire on their first day. They are capable of excellent work, but they need to know:
- **What** exactly you want
- **Who** the audience is
- **How** to format the output
- **How long** the output should be
- **What style/tone** to use

### The pattern
```
BAD:  "Summarize this article."
GOOD: "Summarize this article in 3 bullet points for a technical audience.
       Each bullet should be one sentence. Use precise terminology.
       Focus on the methodology and results, not the introduction."
```

Every word of specificity you add reduces ambiguity and improves output quality.

In [ ]:
# The article we'll work with throughout this section
article = """Researchers at MIT have developed a new machine learning model called
NeuroFlex that achieves state-of-the-art performance on protein folding prediction
tasks. The model uses a novel attention mechanism called Rotary Axial Attention (RAA)
that reduces computational complexity from O(n^3) to O(n*log(n)) while maintaining
accuracy. In benchmarks against AlphaFold2, NeuroFlex achieved 94.2% accuracy on the
CASP15 dataset compared to AlphaFold2's 91.8%, while using 60% less compute during
inference. The team trained the model on 2.3 million protein structures from the PDB
database using a curriculum learning approach. The work has implications for drug
discovery, as faster protein folding prediction could accelerate the identification
of therapeutic targets. The researchers plan to release the model weights and code
under an Apache 2.0 license."""

# BAD prompt: vague, no constraints
bad_prompt = f"Summarize this article:\n\n{article}"

# GOOD prompt: specific about format, audience, length, focus
good_prompt = f"""Summarize the following article for a senior ML engineer who needs to decide
whether to read the full paper.

Requirements:
- Exactly 3 bullet points
- Each bullet: one sentence, max 25 words
- Focus on: (1) what the model does, (2) key benchmark result, (3) practical implication
- Use precise technical terminology
- Do NOT include background context or future plans

Article:
{article}"""

compare("BAD: Vague prompt", bad_prompt, "GOOD: Specific prompt", good_prompt)

In [ ]:
# Another example: instruction specificity for a different task
# Notice how each constraint eliminates an entire category of bad output

bad_email = "Write an email to a customer about their late shipment."

good_email = """Write an apology email to a customer whose order #4521 (a laptop) is 5 days late
due to a warehouse logistics issue.

Requirements:
- Tone: professional, empathetic, not overly apologetic
- Length: 4-6 sentences
- Include: specific order number, revised delivery date (3 business days from now), a 15% discount code (SORRY15)
- Do NOT: blame third parties, use the word "unfortunately", or make promises about future orders
- Sign off as "The Support Team"""

compare("BAD: Vague email request", bad_email, "GOOD: Fully specified email", good_email)

### Key takeaway for the interview

When you get a customer scenario, your FIRST move should be to add specificity to any prompt you write:
1. **Format**: bullet points? paragraph? JSON? table?
2. **Length**: max words/sentences/items
3. **Audience**: who will read this output?
4. **Focus**: what to include AND what to exclude
5. **Tone**: formal? casual? technical?

This takes 30 seconds and saves 5 minutes of iteration.

---

## 1.2 Few-Shot Examples (Multishot Prompting)

Examples are worth a thousand words of instructions. When you show Claude what you want, it pattern-matches precisely.

### The principle
- Include 2-5 input/output example pairs in your prompt
- Cover the **typical cases** AND at least one **edge case**
- Use XML tags to clearly delineate examples (Claude responds very well to XML)
- Make examples consistent in format so Claude can extract the pattern

### The pattern
```xml
<examples>
  <example>
    <input>example input 1</input>
    <output>example output 1</output>
  </example>
  <example>
    <input>example input 2</input>
    <output>example output 2</output>
  </example>
</examples>
```

In [ ]:
# Task: Sentiment classification of customer reviews
# WITHOUT examples: Claude guesses at your labeling conventions

no_examples_prompt = """Classify the sentiment of this customer review as positive, negative, or neutral.

Review: "The product works fine but the delivery took forever and the packaging was damaged."""

# WITH few-shot examples: Claude matches your exact format and conventions
few_shot_prompt = """Classify the sentiment of customer reviews. Use exactly one label: POSITIVE, NEGATIVE, or MIXED.
Return ONLY the label, nothing else.

<examples>
<example>
<input>This laptop is incredible. Fast, lightweight, and the battery lasts all day.</input>
<output>POSITIVE</output>
</example>
<example>
<input>Complete waste of money. Broke after two days and customer service was useless.</input>
<output>NEGATIVE</output>
</example>
<example>
<input>The camera quality is great but the phone overheats during video calls.</input>
<output>MIXED</output>
</example>
</examples>

<input>The product works fine but the delivery took forever and the packaging was damaged.</input>"""

compare(
    "WITHOUT examples (guesses format, may use 'neutral')",
    no_examples_prompt,
    "WITH few-shot examples (uses exact label format, MIXED)",
    few_shot_prompt
)

In [ ]:
# Edge case examples dramatically improve accuracy on ambiguous inputs
# Here we add sarcasm and backhanded compliments as edge cases

edge_case_prompt = """Classify the sentiment of customer reviews. Use exactly one label: POSITIVE, NEGATIVE, or MIXED.
Return ONLY the label, nothing else.

<examples>
<example>
<input>This laptop is incredible. Fast, lightweight, and the battery lasts all day.</input>
<output>POSITIVE</output>
</example>
<example>
<input>Complete waste of money. Broke after two days.</input>
<output>NEGATIVE</output>
</example>
<example>
<input>The camera quality is great but the phone overheats during video calls.</input>
<output>MIXED</output>
</example>
<example>
<input>Oh great, another product that "just works" until you actually need it to work.</input>
<output>NEGATIVE</output>
</example>
<example>
<input>It's the best product I've ever used, if you ignore all the other products I've used.</input>
<output>NEGATIVE</output>
</example>
</examples>

Classify these reviews one per line:

1. "Wow, I'm SO impressed by how quickly it stopped working."
2. "Solid product, fair price, does what it says."
3. "I guess it's fine for the price, but don't expect miracles."""

print(ask(edge_case_prompt))

### Key takeaway for the interview

When you need Claude to follow a specific pattern:
1. Write 2-3 examples of the **common** case
2. Add 1-2 examples of **edge cases** (sarcasm, ambiguity, empty input, etc.)
3. Wrap them in `<examples>` XML tags
4. Use consistent formatting across all examples

This is faster than writing paragraphs of instructions about how to handle edge cases.

---

## 1.3 Chain of Thought (CoT)

For complex reasoning tasks, asking Claude to think step-by-step before answering dramatically improves accuracy.

### The principle
- Without CoT, Claude jumps straight to an answer and may skip critical reasoning steps
- With CoT, Claude works through the problem systematically, catching errors along the way
- Two levels: simple ("Think step by step") and structured (XML-tagged thinking)

### When to use it
- Math or logic problems
- Multi-step analysis
- Tasks where the answer depends on weighing multiple factors
- Classification with complex criteria

### When NOT to use it
- Simple factual questions
- Creative writing
- Tasks where you want concise output (CoT adds length)

In [ ]:
# A reasoning problem that often fails without CoT
reasoning_problem = """A store has a promotion: buy 3 items, get the cheapest one free.
Sarah buys items priced at $45, $30, $25, $20, and $15.
She wants to minimize her total cost.
What grouping should she use and what will she pay?"""

# WITHOUT Chain of Thought
no_cot = reasoning_problem

# WITH simple CoT
simple_cot = reasoning_problem + "\n\nThink through this step by step before giving your final answer."

compare("WITHOUT CoT (may rush to wrong answer)", no_cot, "WITH simple CoT", simple_cot)

In [ ]:
# STRUCTURED CoT with XML tags — gives you the best of both worlds:
# Claude reasons thoroughly, but you can easily parse the final answer

structured_cot_prompt = f"""Solve the following problem. Show your reasoning inside <thinking> tags,
then give your final answer inside <answer> tags.

<problem>
{reasoning_problem}
</problem>

Use this format:
<thinking>
[Your step-by-step reasoning here]
</thinking>

<answer>
[Final answer with the grouping and total cost]
</answer>"""

result = ask(structured_cot_prompt, max_tokens=2048)
print(result)

# Parse just the answer if needed
import re
answer_match = re.search(r'<answer>(.*?)</answer>', result, re.DOTALL)
if answer_match:
    print("\n" + "=" * 40)
    print("PARSED ANSWER:")
    print(answer_match.group(1).strip())

In [ ]:
# Another great CoT use case: complex classification
# This is common in customer scenario interviews

ticket_classification = """Classify this support ticket into exactly one category.
Categories: BILLING, TECHNICAL, ACCOUNT, FEATURE_REQUEST, SPAM

Think through your reasoning in <thinking> tags, then give the classification in <category> tags.

<ticket>
Subject: Can't access my dashboard after upgrading
Body: Hi, I upgraded to the Pro plan yesterday and was charged $49.99. But now when I try
to log into my dashboard, I get a 403 error. I've tried clearing cookies and using a different
browser. My subscription shows as active in the billing portal. Can someone help?
</ticket>

<thinking>
[Analyze: what is the primary issue? what category best fits?]
</thinking>

<category>[ONE WORD]</category>"""

result = ask(ticket_classification)
print(result)

# Note: This ticket could plausibly be BILLING or TECHNICAL.
# CoT forces Claude to reason about which is PRIMARY, leading to more consistent results.

### Key takeaway for the interview

- For simple tasks: skip CoT (it adds latency and tokens)
- For reasoning tasks: add "Think step by step" at minimum
- For tasks where you need both reasoning AND a parseable answer: use `<thinking>` and `<answer>` XML tags
- The structured format is especially useful when you need to pipe the answer into the next step of a chain

---

## 1.4 XML Tags for Structure

This is arguably the most important technique for Claude specifically. Claude was trained to respond very well to XML-structured prompts. Anthropic's own documentation uses XML extensively.

### Why XML?
- Clearly separates different parts of the prompt (instructions vs. context vs. examples)
- Eliminates ambiguity about where user data begins and instructions end
- Makes prompts modular and easy to modify
- Claude can be asked to output XML, making parsing trivial

### Core tags to memorize
| Tag | Purpose |
|-----|--------|
| `<instructions>` | What Claude should do |
| `<context>` | Background information |
| `<examples>` | Few-shot examples |
| `<input>` | The actual data to process |
| `<output_format>` | How to structure the response |
| `<constraints>` | Rules and restrictions |
| `<thinking>` | For CoT reasoning |
| `<document>` | Long documents or data |

### The template
```xml
<instructions>
What to do
</instructions>

<context>
Background info
</context>

<constraints>
Rules and restrictions
</constraints>

<output_format>
How to format the response
</output_format>

<input>
The actual data
</input>
```

In [ ]:
# Example: A complete XML-structured prompt for a realistic task
# Task: Extract action items from meeting notes

meeting_notes = """Team standup - Jan 15, 2025

Alice: The authentication module is almost done. I need to add rate limiting
before Friday's release. Also, the QA team found a bug in the password reset
flow -- I'll fix that today.

Bob: I finished the database migration script. It needs a code review from Alice
or Charlie before we can run it on staging. I'll start working on the API
documentation after lunch.

Charlie: The client demo is Thursday at 2pm. I need the auth module deployed to
staging by Wednesday EOD so I can do a dry run. Also, we should schedule a
retrospective for next week -- can someone set that up?

Manager: Let's make sure we have test coverage above 80% before the release.
Alice, can you check the current coverage numbers?"""

xml_prompt = f"""<instructions>
Extract all action items from the meeting notes below. For each action item, identify
the owner, the task, the deadline (if mentioned), and the priority (high/medium/low).
</instructions>

<context>
These are daily standup notes from a software engineering team preparing for a
Friday product release. The client demo is Thursday.
</context>

<constraints>
- Only include explicit action items, not general status updates
- If no deadline is mentioned, write "Not specified"
- Priority should be based on the release timeline:
  HIGH = blocks the release or demo
  MEDIUM = should be done this week
  LOW = can wait until next week
- If the owner is unclear, write "Unassigned"
</constraints>

<output_format>
Return a markdown table with columns: Owner | Task | Deadline | Priority
Sort by priority (HIGH first), then by deadline.
</output_format>

<input>
{meeting_notes}
</input>"""

print(ask(xml_prompt, max_tokens=2048))

In [ ]:
# Compare: the same task without XML structure
# This works, but the output is less consistent and harder to control

flat_prompt = f"""Extract action items from these meeting notes. Include who owns each item,
what the task is, the deadline, and priority.

{meeting_notes}"""

compare(
    "FLAT prompt (works but inconsistent)",
    flat_prompt,
    "XML-STRUCTURED prompt (consistent, parseable)",
    xml_prompt,
    max_tokens=2048
)

In [ ]:
# XML also works brilliantly for separating multiple documents or inputs
# This is critical when your prompt includes user-provided content that
# could be confused with instructions (prompt injection defense)

multi_doc_prompt = """<instructions>
Compare the two product reviews below. Identify:
1. Points of agreement
2. Points of disagreement
3. Which review is more helpful and why (one sentence)
</instructions>

<review id="1">
The XPS 15 has an excellent 3.5K OLED display and the keyboard is top-tier.
However, the battery life is disappointing at 6 hours and it runs hot under
sustained load. The 32GB RAM config is overpriced at $2,200.
</review>

<review id="2">
I've been using the XPS 15 for 3 months. Display is stunning -- best I've
ever used on a laptop. Battery gets me through about 7 hours with light use.
Thermals are an issue during video editing, but fine for everyday tasks.
Great value at the $1,800 base config.
</review>

<output_format>
Use headers: ## Agreement, ## Disagreement, ## Verdict
Keep each section to 2-3 bullet points max.
</output_format>"""

print(ask(multi_doc_prompt))

### Key takeaway for the interview

Use XML tags in EVERY non-trivial prompt. The template to reach for:

```xml
<instructions>What to do</instructions>
<context>Background</context>
<constraints>Rules</constraints>
<output_format>How to format</output_format>
<input>The data</input>
```

You do not need all five tags every time. Use the ones that apply. But `<instructions>` and `<input>` should be in almost every prompt.

---

## 1.5 System Prompts and Roles

System prompts set the behavioral foundation for the entire conversation. They define WHO Claude is, HOW it should behave, and WHAT rules it should follow.

### System prompt anatomy
1. **Role definition**: Who are you?
2. **Expertise**: What do you know?
3. **Behavioral rules**: How should you act?
4. **Output format**: How should you respond?
5. **Guardrails**: What should you never do?

### Quick template (memorize this)
```
You are a [ROLE] with expertise in [DOMAIN].

Your task is to [PRIMARY GOAL].

Rules:
- [Rule 1]
- [Rule 2]
- [Rule 3]

Response format: [FORMAT SPEC]
```

In [ ]:
# Same question, two different system prompts — dramatically different output
# This demonstrates how the system prompt shapes everything

question = "Should our startup switch from PostgreSQL to MongoDB for our e-commerce platform?"

# System prompt A: Technical architect
system_architect = """You are a senior database architect with 15 years of experience in both SQL and NoSQL systems.

Your task is to provide technical advice on database selection.

Rules:
- Always consider data modeling requirements, consistency needs, and query patterns
- Provide concrete technical trade-offs, not vague generalities
- Be direct -- recommend one option and explain why
- Include at least one risk/mitigation for your recommendation

Response format: 2-3 short paragraphs. No bullet points. Technical but accessible."""

# System prompt B: Startup advisor
system_advisor = """You are a startup CTO advisor who has helped 50+ early-stage companies make technology decisions.

Your task is to advise on technology choices considering both technical and business factors.

Rules:
- Always consider team skill set, development speed, hiring implications, and cost
- Prioritize shipping speed over theoretical perfection
- Be opinionated but acknowledge the main counterargument
- Frame advice in terms of business impact, not just technical merit

Response format: Start with a one-sentence recommendation. Then 3-4 bullet points."""

print("=" * 60)
print("  SYSTEM PROMPT A: Database Architect")
print("=" * 60)
print(ask(question, system=system_architect))
print("\n" + "=" * 60)
print("  SYSTEM PROMPT B: Startup Advisor")
print("=" * 60)
print(ask(question, system=system_advisor))

In [ ]:
# Real interview pattern: system prompt for a customer-facing assistant
# This is the kind of system prompt you'd build in the interview

customer_service_system = """You are a customer support assistant for TechCorp, a SaaS company selling project management software.

Your task is to help customers resolve issues with their accounts and software.

Rules:
- Be friendly, professional, and concise
- If you cannot resolve an issue, explain what you've tried and escalate to a human agent
- Never share internal system details, pricing negotiations, or other customers' information
- Never make up information about product features -- if unsure, say so
- Always confirm the customer's issue before providing a solution
- If the customer seems frustrated, acknowledge their feelings before problem-solving

Available actions (use these in your responses):
- [CHECK_STATUS]: Reference an order or subscription status
- [RESET_PASSWORD]: Trigger a password reset email
- [ESCALATE]: Hand off to a human agent with context
- [APPLY_CREDIT]: Apply account credit (max $50 without manager approval)

Response format: Keep responses under 150 words. Use short paragraphs."""

# Test with an angry customer
angry_customer = """This is ridiculous! I've been paying $49/month for 6 months and half the features
don't even work. The Gantt chart view has been broken for weeks and nobody seems to care.
I want a refund or I'm switching to Asana."""

print(ask(angry_customer, system=customer_service_system))

In [ ]:
# Test the same system prompt with a simple question
simple_question = "How do I add a new team member to my project?"

print(ask(simple_question, system=customer_service_system))

### Key takeaway for the interview

System prompts should be your FIRST step when building any customer-facing solution. Use the template:

```
You are [ROLE] for [COMPANY/PRODUCT].
Your task is [PRIMARY GOAL].
Rules: [3-5 behavioral constraints]
Response format: [length/style spec]
```

Build it in 60 seconds, then iterate as you test.

---

## 1.6 Prompt Chaining

Complex tasks almost always produce better results when broken into sequential subtasks, where the output of one step becomes the input to the next.

### The principle
- Each step does ONE thing well
- The output of step N feeds into step N+1
- Each step can have its own optimized prompt
- If one step fails, you can re-run just that step

### Common chain patterns
- **Extract -> Transform -> Format**: Pull data, process it, format for output
- **Analyze -> Decide -> Act**: Understand the situation, make a choice, take action
- **Draft -> Critique -> Revise**: Generate content, evaluate it, improve it
- **Classify -> Route -> Respond**: Categorize input, choose handler, generate response

In [ ]:
# Prompt chain: Document -> Extract Key Points -> Generate Summary -> Format as Email
# Each step is a separate API call with a focused prompt

document = """Q4 2024 Engineering Report

Platform Reliability: We achieved 99.95% uptime this quarter, up from 99.91% in Q3.
The main contributor was the migration to redundant load balancers in October. We had
one P1 incident on Nov 12 (47 minutes of downtime) caused by a database connection
pool exhaustion. Post-mortem action items have been completed.

Feature Delivery: The team shipped 23 features this quarter, including the new
real-time collaboration module (Project Atlas), advanced search with filters, and
the mobile app v2.0 launch. Project Atlas alone required 3 months of development
and involved 8 engineers.

Performance: API p95 latency dropped from 340ms to 180ms after the caching layer
overhaul. Database query optimization reduced average query time by 40%. The frontend
bundle size decreased by 25% through code splitting.

Team: We hired 4 new engineers (2 senior, 2 mid-level). Attrition was 1 departure
(voluntary, relocated). Team satisfaction score: 4.2/5.0 (up from 3.8 in Q3).
Main concern from the team survey: need for better on-call rotation.

Q1 2025 Priorities: (1) Launch customer-facing API, (2) SOC2 compliance preparation,
(3) Reduce P1 incident response time to under 15 minutes."""

# STEP 1: Extract key points
step1_prompt = f"""<instructions>
Extract the 5 most important facts from this engineering report.
Each fact should be a single sentence with a specific number or metric.
Return ONLY the facts as a numbered list.
</instructions>

<document>
{document}
</document>"""

key_points = ask(step1_prompt)
print("STEP 1 - Key Points:")
print(key_points)
print()

In [ ]:
# STEP 2: Generate executive summary from key points
step2_prompt = f"""<instructions>
Write a 2-sentence executive summary based on these key points.
The first sentence should cover the biggest win.
The second sentence should cover the top priority going forward.
Write for a non-technical executive audience.
</instructions>

<key_points>
{key_points}
</key_points>"""

summary = ask(step2_prompt)
print("STEP 2 - Executive Summary:")
print(summary)
print()

In [ ]:
# STEP 3: Format as email
step3_prompt = f"""<instructions>
Format the following executive summary and key points as a brief email
from the VP of Engineering to the CEO.
</instructions>

<summary>
{summary}
</summary>

<key_points>
{key_points}
</key_points>

<constraints>
- Subject line should be concise and action-oriented
- Open with the summary
- Include key points as bullet points under "Highlights:"
- Close with "Happy to discuss in more detail at our 1:1."
- Total length: under 200 words
- Sign as "Jamie, VP Engineering"
</constraints>"""

email = ask(step3_prompt)
print("STEP 3 - Final Email:")
print(email)

In [ ]:
# Reusable prompt chaining helper
# This is a pattern you can use in the interview to quickly chain steps

def chain(steps, initial_input, verbose=True):
    """Execute a sequence of prompt steps, piping output to input.
    
    Args:
        steps: List of dicts with 'name' and 'prompt_template' (uses {input} placeholder)
        initial_input: The starting input text
        verbose: Print intermediate results
    
    Returns:
        Final output text
    """
    current_input = initial_input
    
    for i, step in enumerate(steps):
        prompt = step["prompt_template"].format(input=current_input)
        system = step.get("system", None)
        current_input = ask(prompt, system=system)
        
        if verbose:
            print(f"--- Step {i+1}: {step['name']} ---")
            print(current_input[:300] + ("..." if len(current_input) > 300 else ""))
            print()
    
    return current_input

# Example: Classify -> Route -> Respond chain for customer support
support_chain = [
    {
        "name": "Classify",
        "prompt_template": """Classify this support ticket into exactly one category.
Categories: BILLING, TECHNICAL, ACCOUNT, FEATURE_REQUEST
Return ONLY the category name.

<ticket>{input}</ticket>"""
    },
    {
        "name": "Generate Response",
        "prompt_template": """The following support ticket has been classified as: {input}

Generate an appropriate first response to the customer. Be empathetic, acknowledge
the specific issue, and provide a concrete next step. Keep it under 100 words.

If BILLING: mention you can check their account and offer to review charges.
If TECHNICAL: ask for browser/OS info and suggest basic troubleshooting.
If ACCOUNT: offer to verify identity and help with account access.
If FEATURE_REQUEST: thank them, explain the feedback process, and provide a timeline if possible."""
    }
]

ticket = """I was charged twice for my subscription this month. Order IDs: #8891 and #8892.
Both show $29.99 on my credit card statement. Please refund the duplicate charge."""

result = chain(support_chain, ticket)

### Key takeaway for the interview

When faced with a complex task:
1. Break it into 2-4 sequential steps
2. Each step gets its own focused prompt
3. Pipe output to input using a simple loop or helper function
4. This is more reliable than trying to do everything in one massive prompt

Common interview chains:
- **Classify -> Route -> Respond** (customer support)
- **Extract -> Validate -> Transform** (data processing)
- **Draft -> Critique -> Revise** (content generation)

---

## 1.7 Structured Output

Getting Claude to output reliable, parseable structured data (JSON, tables, specific formats). This is critical when Claude's output needs to feed into code.

### Three approaches (from simplest to most reliable)

1. **Ask nicely in the prompt** -- works 90% of the time
2. **Use XML output tags** -- works 98% of the time
3. **Use `tool_choice` to force structured output** -- works 100% of the time (see Notebook 1/4)

In [ ]:
# Approach 1: Ask for JSON in the prompt
# Works well for simple structures, but Claude may add explanation text

json_prompt = """Extract the following information from this text and return it as valid JSON.

Text: "John Smith, age 34, works at Google as a Senior Engineer in Mountain View, CA.
He's been there for 5 years and previously worked at Meta. His email is john.smith@gmail.com."

Return JSON with these fields:
- name (string)
- age (integer)
- company (string)
- title (string)
- location (string)
- tenure_years (integer)
- previous_company (string)
- email (string)

Return ONLY the JSON object, no explanation or markdown formatting."""

result = ask(json_prompt)
print("Raw output:")
print(result)

# Try to parse it
try:
    # Strip markdown code fences if Claude adds them
    clean = result.strip()
    if clean.startswith("```"):
        clean = clean.split("\n", 1)[1].rsplit("```", 1)[0]
    data = json.loads(clean)
    print("\nParsed successfully:")
    print(json.dumps(data, indent=2))
except json.JSONDecodeError as e:
    print(f"\nFailed to parse: {e}")

In [ ]:
# Approach 2: XML tags for structured output
# More reliable because Claude clearly knows where the data boundaries are

xml_output_prompt = """<instructions>
Extract structured data from the text below.
Return the data inside <json> tags as valid JSON.
Do not include any text outside the <json> tags.
</instructions>

<schema>
{
  "name": "string",
  "age": "integer",
  "company": "string",
  "title": "string",
  "location": "string",
  "tenure_years": "integer",
  "previous_company": "string",
  "email": "string"
}
</schema>

<input>
John Smith, age 34, works at Google as a Senior Engineer in Mountain View, CA.
He's been there for 5 years and previously worked at Meta. His email is john.smith@gmail.com.
</input>"""

result = ask(xml_output_prompt)
print("Raw output:")
print(result)

# Parse from XML tags
json_match = re.search(r'<json>(.*?)</json>', result, re.DOTALL)
if json_match:
    data = json.loads(json_match.group(1).strip())
    print("\nParsed from XML tags:")
    print(json.dumps(data, indent=2))

In [ ]:
# Approach 3: tool_choice for guaranteed structured output (100% reliable)
# This uses the tool use mechanism as a structured output schema
# Claude is FORCED to return data matching the tool's input_schema

extract_person_tool = {
    "name": "extract_person",
    "description": "Extract person information from text. Call this with the extracted data.",
    "input_schema": {
        "type": "object",
        "properties": {
            "name": {"type": "string", "description": "Full name"},
            "age": {"type": "integer", "description": "Age in years"},
            "company": {"type": "string", "description": "Current employer"},
            "title": {"type": "string", "description": "Job title"},
            "location": {"type": "string", "description": "City, State"},
            "tenure_years": {"type": "integer", "description": "Years at current company"},
            "previous_company": {"type": "string", "description": "Previous employer"},
            "email": {"type": "string", "description": "Email address"}
        },
        "required": ["name", "age", "company", "title", "location", "tenure_years", "previous_company", "email"]
    }
}

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=[extract_person_tool],
    tool_choice={"type": "tool", "name": "extract_person"},  # FORCE this tool
    messages=[{
        "role": "user",
        "content": """John Smith, age 34, works at Google as a Senior Engineer in Mountain View, CA.
He's been there for 5 years and previously worked at Meta. His email is john.smith@gmail.com."""
    }]
)

# The tool input IS the structured data -- guaranteed to match the schema
tool_block = next(b for b in response.content if b.type == "tool_use")
data = tool_block.input

print("Extracted via tool_choice (guaranteed structure):")
print(json.dumps(data, indent=2))
print(f"\nType of 'age': {type(data['age']).__name__}")  # Guaranteed integer!

In [ ]:
# Real interview use case: extracting multiple entities from a complex text

extract_order_tool = {
    "name": "extract_order_info",
    "description": "Extract order information from a customer message. Call this with the extracted data.",
    "input_schema": {
        "type": "object",
        "properties": {
            "customer_name": {"type": "string", "description": "Customer's name if mentioned"},
            "order_ids": {
                "type": "array",
                "items": {"type": "string"},
                "description": "All order IDs mentioned"
            },
            "issue_type": {
                "type": "string",
                "enum": ["refund", "shipping", "damaged", "wrong_item", "billing", "other"],
                "description": "Primary issue category"
            },
            "urgency": {
                "type": "string",
                "enum": ["low", "medium", "high"],
                "description": "How urgent this appears based on language and situation"
            },
            "amount_mentioned": {"type": "number", "description": "Dollar amount if mentioned"},
            "summary": {"type": "string", "description": "One-sentence summary of the issue"}
        },
        "required": ["order_ids", "issue_type", "urgency", "summary"]
    }
}

customer_message = """Hi, this is Maria Rodriguez. I placed two orders last week -- #ORD-4521 and
#ORD-4523. The first one arrived but the laptop was damaged (cracked screen). The second one
hasn't shipped yet even though I paid $89 for overnight shipping. I need the laptop for a
presentation on Monday. This is really urgent. Total charges were $1,299 and $89."""

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=[extract_order_tool],
    tool_choice={"type": "tool", "name": "extract_order_info"},
    messages=[{"role": "user", "content": customer_message}]
)

order_info = next(b for b in response.content if b.type == "tool_use").input
print("Extracted order info:")
print(json.dumps(order_info, indent=2))

### Key takeaway for the interview

Use the right approach based on reliability needs:

| Approach | Reliability | Speed | When to use |
|----------|------------|-------|-------------|
| Ask in prompt | ~90% | Fastest to write | Quick prototyping, human-readable output |
| XML output tags | ~98% | Medium | When you need to parse the output in code |
| `tool_choice` forced | ~100% | Most code | When structured data feeds into next step |

In the interview, if you need reliable structured output, go straight to `tool_choice`. It is worth the extra 30 seconds of setup.

---

## 1.8 The Iterative Improvement Exercise

This section simulates what you will actually do in the interview: start with a basic prompt, then systematically improve it by layering techniques.

### The scenario
A customer wants an AI assistant that processes customer feedback emails and generates:
1. A sentiment label (positive/negative/mixed)
2. Key issues mentioned
3. A suggested response draft

We will start with the simplest possible prompt and improve it step by step.

In [ ]:
# The test input -- a realistic customer email
customer_email = """Subject: Disappointed with recent experience

Hi,

I've been a loyal customer for 3 years and I'm really disappointed with my recent
experience. I ordered the Premium Plan upgrade on January 5th and was charged $199.
However:

1. The advanced analytics dashboard still shows the free tier version
2. The API rate limit hasn't increased as promised (still at 100 req/min instead of 1000)
3. I can't access the priority support chat -- it says "Feature not available"

I've already emailed support twice (Jan 7 and Jan 10) with no response. This is
unacceptable for a paid plan.

That said, I do love the core product and the new reporting features are great.
I just need these Premium features to actually work.

Please fix this ASAP or refund my upgrade.

Thanks,
David Chen
Account: PRO-8834"""

print("Test email loaded. Length:", len(customer_email), "chars")

In [ ]:
# ITERATION 0: The naive prompt
# This is what someone with no prompt engineering training would write

v0_prompt = f"""Analyze this customer email and tell me the sentiment, key issues, and
write a response.

{customer_email}"""

print("=== V0: Naive Prompt ===")
print(ask(v0_prompt, max_tokens=2048))

In [ ]:
# ITERATION 1: Add clear instructions (Section 1.1)
# Specify format, length, and what we actually need

v1_prompt = f"""Analyze this customer email. Provide:

1. Sentiment: exactly one of POSITIVE, NEGATIVE, or MIXED
2. Key Issues: a numbered list of specific problems mentioned (max 5)
3. Suggested Response: a professional email reply (under 150 words)

Use clear headers for each section.
In the response, address each issue specifically and provide next steps.
Do not be overly apologetic -- be professional and solution-oriented.

Customer email:
{customer_email}"""

print("=== V1: Clear Instructions ===")
print(ask(v1_prompt, max_tokens=2048))

In [ ]:
# ITERATION 2: Add XML structure (Section 1.4)
# Separate the instructions, constraints, and input clearly

v2_prompt = f"""<instructions>
Analyze the customer email below and produce three outputs:
1. Sentiment classification
2. List of key issues
3. Suggested response email
</instructions>

<constraints>
- Sentiment must be exactly one of: POSITIVE, NEGATIVE, MIXED
- List at most 5 key issues, each as one sentence
- Response email must be under 150 words
- Response tone: professional, empathetic, solution-oriented
- Response must address each specific issue mentioned
- Response must include a concrete next step with timeline
- Do not use the word "unfortunately" or "sincerely"
</constraints>

<output_format>
## Sentiment
[LABEL]

## Key Issues
1. [issue]
2. [issue]
...

## Suggested Response
[email text]
</output_format>

<input>
{customer_email}
</input>"""

print("=== V2: XML Structure ===")
print(ask(v2_prompt, max_tokens=2048))

In [ ]:
# ITERATION 3: Add few-shot examples (Section 1.2)
# Show Claude exactly what we want the output to look like

v3_prompt = f"""<instructions>
Analyze customer emails and produce a sentiment classification, key issues list,
and a suggested response email.
</instructions>

<constraints>
- Sentiment: exactly one of POSITIVE, NEGATIVE, MIXED
- Key issues: max 5, each one sentence
- Response: under 150 words, professional, empathetic, solution-oriented
- Response must address each issue and include next steps with timeline
</constraints>

<examples>
<example>
<email>
Subject: Love the new update!
The new dashboard is amazing. One small thing -- the export button seems broken on Firefox.
Otherwise, great work!
-- Sarah M, Account: FREE-2201
</email>
<analysis>
## Sentiment
MIXED

## Key Issues
1. Export button broken on Firefox browser

## Suggested Response
Hi Sarah,

Thank you for the kind words about the new dashboard! We're glad you're enjoying it.

Regarding the Firefox export issue -- we've flagged this with our engineering team. A fix
will be deployed within 48 hours. In the meantime, the export works correctly in Chrome
and Edge as a workaround.

We'll follow up once the fix is live.

Best,
The Support Team
</analysis>
</example>
</examples>

<input>
{customer_email}
</input>"""

print("=== V3: XML + Few-Shot Examples ===")
print(ask(v3_prompt, max_tokens=2048))

In [ ]:
# ITERATION 4: Add system prompt (Section 1.5)
# Set the role and behavioral foundation

v4_system = """You are a senior customer success specialist at a SaaS company.

Your task is to analyze customer feedback emails and prepare response drafts for the support team.

Rules:
- Always validate the customer's frustration before jumping to solutions
- Reference their loyalty/tenure if mentioned (it matters to retention)
- Provide specific timelines, not vague promises
- Mention the customer's name and account number in the response
- If a refund is mentioned, acknowledge it but offer to resolve first
- Escalation path: if 2+ issues are unresolved after initial contact, flag for manager review"""

# Same prompt as V3, but now with the system prompt providing role context
v4_prompt = v3_prompt  # Reuse the XML + few-shot prompt

print("=== V4: System Prompt + XML + Few-Shot ===")
print(ask(v4_prompt, system=v4_system, max_tokens=2048))

In [ ]:
# ITERATION 5 (FINAL): Structured output via tool_choice for machine-readable results
# This is the production-grade version

analyze_email_tool = {
    "name": "analyze_email",
    "description": "Analyze a customer email. Extract sentiment, issues, and generate a response.",
    "input_schema": {
        "type": "object",
        "properties": {
            "sentiment": {
                "type": "string",
                "enum": ["POSITIVE", "NEGATIVE", "MIXED"],
                "description": "Overall sentiment of the email"
            },
            "key_issues": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "issue": {"type": "string", "description": "One-sentence description"},
                        "severity": {"type": "string", "enum": ["low", "medium", "high"]},
                        "category": {"type": "string", "enum": ["billing", "technical", "support", "feature"]}
                    },
                    "required": ["issue", "severity", "category"]
                },
                "description": "List of issues mentioned in the email"
            },
            "customer_name": {"type": "string", "description": "Customer name if mentioned"},
            "account_id": {"type": "string", "description": "Account ID if mentioned"},
            "needs_escalation": {"type": "boolean", "description": "Whether this needs manager review"},
            "suggested_response": {"type": "string", "description": "Draft response email under 150 words"}
        },
        "required": ["sentiment", "key_issues", "needs_escalation", "suggested_response"]
    }
}

response = client.messages.create(
    model=MODEL,
    max_tokens=2048,
    system=v4_system,
    tools=[analyze_email_tool],
    tool_choice={"type": "tool", "name": "analyze_email"},
    messages=[{"role": "user", "content": customer_email}]
)

analysis = next(b for b in response.content if b.type == "tool_use").input

print("=== V5: Full Structured Output (tool_choice) ===")
print(json.dumps(analysis, indent=2))

# Now you can use this data programmatically
print(f"\n--- Programmatic access ---")
print(f"Sentiment: {analysis['sentiment']}")
print(f"Issues found: {len(analysis['key_issues'])}")
print(f"Needs escalation: {analysis['needs_escalation']}")
print(f"Customer: {analysis.get('customer_name', 'Unknown')}")
print(f"Account: {analysis.get('account_id', 'Unknown')}")

### Improvement progression summary

| Version | Technique Added | What Improved |
|---------|----------------|---------------|
| V0 | None (naive) | Baseline -- output is unstructured and inconsistent |
| V1 | Clear instructions | Output has the right sections; format is more consistent |
| V2 | XML structure | Instructions, constraints, and input are clearly separated |
| V3 | Few-shot examples | Output format matches exactly what we want |
| V4 | System prompt | Tone and behavioral nuances are correct (empathy, loyalty mention) |
| V5 | tool_choice | Output is guaranteed-parseable JSON with typed fields |

**In the interview, you will NOT have time for all 5 iterations.** Aim for V2-V4 quality depending on the task. Use V5 (tool_choice) when the output must feed into code.

---

## Summary: Technique Priority Order

When you are under time pressure in the interview, reach for techniques in this order:

### Priority 1: Always Do These (30 seconds each)
1. **Clear instructions** -- specify format, length, audience, focus
2. **XML tags** -- separate `<instructions>`, `<input>`, and `<constraints>` at minimum

### Priority 2: Do If You Have Time (60 seconds each)
3. **System prompt** -- set role and behavioral rules
4. **Few-shot examples** -- 2-3 examples for any classification or formatting task

### Priority 3: Use When Needed
5. **Chain of Thought** -- for reasoning, math, complex classification
6. **Prompt chaining** -- when the task has clear sequential steps
7. **Structured output (tool_choice)** -- when output feeds into code

### The "5 Minutes Left" Quick-Win Checklist

If your prompt is not working and you have 5 minutes left:

- [ ] **Did you specify the output format?** ("Return as JSON", "Use bullet points", etc.)
- [ ] **Did you separate instructions from data?** (Use `<instructions>` and `<input>` tags)
- [ ] **Did you add constraints?** ("Do not...", "Max 5 items", "One sentence each")
- [ ] **Did you add an example?** (Even ONE example dramatically improves consistency)
- [ ] **Did you add a system prompt?** (Role + rules takes 30 seconds)
- [ ] **Is the task too complex for one prompt?** (Split into 2 chained calls)

### Quick Reference: XML Tag Cheat Sheet
```xml
<instructions>What to do</instructions>
<context>Background info</context>
<examples><example><input>...</input><output>...</output></example></examples>
<constraints>Rules and restrictions</constraints>
<output_format>How to format the response</output_format>
<input>The actual data to process</input>
<thinking>For CoT reasoning</thinking>
<answer>For CoT final answer</answer>
```

---

**Next: Move to the customer_scenarios/ notebooks for full interview simulations that combine prompt engineering with tool use and agentic patterns.**